In [ ]:

# Session Reload

from google.colab import drive
drive.mount('/content/drive')

import os, warnings
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
import numpy as np
import tensorflow as tf
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

PROJECT_ROOT = '/content/drive/MyDrive/MRI_Brain_Tumor_Project'
CLASS_NAMES  = ['glioma', 'meningioma', 'notumor', 'pituitary']
PRED_DIR     = f'{PROJECT_ROOT}/results/predictions'

effnet_model = tf.keras.models.load_model(
    f'{PROJECT_ROOT}/models/checkpoints/effnet_correct_s2b.keras'
)

y_true        = np.load(f'{PRED_DIR}/y_true.npy')
y_pred_effnet = np.load(f'{PRED_DIR}/y_pred_effnet_final.npy')
uncertainty   = np.load(f'{PRED_DIR}/uncertainty.npy')

print(f"GPU: {tf.config.list_physical_devices('GPU')}")
print(f"Model: {effnet_model.name}")
print(f"Overall acc: {np.mean(y_pred_effnet == y_true):.4f}")


Mounted at /content/drive
GPU: []
Model: efficientnetb3_correct
Overall acc: 0.9144


In [ ]:

#  Install Gradio (compatible version)
!pip install -q gradio huggingface_hub --upgrade

import gradio as gr
import numpy as np
import tensorflow as tf
import cv2
import os
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = '/content/drive/MyDrive/MRI_Brain_Tumor_Project'
CLASS_NAMES  = ['glioma', 'meningioma', 'notumor', 'pituitary']
IMG_SIZE     = (224, 224)

# Uncertainty thresholds (Task 38)
UNC_HIGH   = 0.15   # red warning — std > 0.15
UNC_LOW    = 0.05   # green confident — std < 0.05

# Load backbone layers for MC Dropout + Grad-CAM++
backbone      = effnet_model.layers[1]
last_conv     = [l.name for l in reversed(backbone.layers)
                 if isinstance(l, tf.keras.layers.Conv2D)][0]
grad_model    = tf.keras.Model(
    inputs=backbone.input,
    outputs=[backbone.get_layer(last_conv).output, backbone.output]
)
dropout_layer = effnet_model.layers[2]
head_layer    = effnet_model.layers[3]

print(f"Gradio version: {gr.__version__}")
print(f"Last conv layer: {last_conv}")
print(f"Uncertainty thresholds: green<{UNC_LOW}, red>{UNC_HIGH}")

#  Helper functions
def preprocess_image(img):
    img_resized = cv2.resize(img, IMG_SIZE)
    img_pre     = tf.keras.applications.efficientnet.preprocess_input(
                      img_resized.astype(np.float32))
    return img_resized, img_pre[np.newaxis]

def mc_dropout_predict(img_pre, n_passes=50):
    features = backbone(img_pre, training=False).numpy()
    feat_tf  = tf.constant(features)
    preds = []
    for _ in range(n_passes):
        drop = dropout_layer(feat_tf, training=True)
        pred = head_layer(drop, training=False).numpy()
        preds.append(pred[0])
    preds       = np.array(preds)
    mc_mean     = preds.mean(axis=0)
    mc_std      = preds.std(axis=0)
    uncertainty = float(mc_std.mean())
    confidence  = float(mc_mean.max())
    pred_class  = int(np.argmax(mc_mean))
    return mc_mean, uncertainty, confidence, pred_class

def make_gradcam(img_pre, pred_class, img_display):
    with tf.GradientTape() as t2:
        with tf.GradientTape() as t1:
            with tf.GradientTape() as t0:
                inp = tf.cast(img_pre, tf.float32)
                co, pred = grad_model(inp)
                t0.watch(co); t1.watch(co); t2.watch(co)
                loss = pred[:, pred_class]
            g1 = t0.gradient(loss, co)
        g2 = t1.gradient(g1, co)
    g3 = t2.gradient(g2, co)
    co=co[0]; g1=g1[0]; g2=g2[0]; g3=g3[0]
    s   = tf.reduce_sum(co, axis=(0,1))
    den = 2*g2 + s[None,None,:]*g3
    den = tf.where(den==0, tf.ones_like(den), den)
    w   = tf.reduce_sum((g2/den)*tf.nn.relu(g1), axis=(0,1))
    h   = tf.nn.relu(tf.reduce_sum(w*co, axis=-1)).numpy()
    h   = cv2.resize(h, IMG_SIZE)
    if h.max() > 0:
        h = (h-h.min())/(h.max()-h.min())
    colormap = matplotlib.colormaps['jet']
    hc  = (colormap(h)[:,:,:3]*255).astype(np.uint8)
    overlay = cv2.addWeighted(img_display, 0.55, hc, 0.45, 0)
    return overlay

def predict_mri(image):
    if image is None:
        return None, "Please upload an MRI image.", ""

    img_display, img_pre = preprocess_image(image)
    mc_mean, unc, conf, pred_class = mc_dropout_predict(img_pre, n_passes=50)
    overlay = make_gradcam(img_pre, pred_class, img_display)

    # Task 38: Uncertainty flag
    if unc > UNC_HIGH:
        flag = f"RED HIGH UNCERTAINTY (std={unc:.4f}) — Radiologist review REQUIRED"
    elif unc < UNC_LOW:
        flag = f"GREEN LOW UNCERTAINTY (std={unc:.4f}) — Model is confident"
    else:
        flag = f"YELLOW MODERATE UNCERTAINTY (std={unc:.4f}) — Consider radiologist review"

    result_text = f"""## Prediction: {CLASS_NAMES[pred_class].upper()}

**Confidence:** {conf:.1%}
**MC Dropout Uncertainty:** {unc:.4f} (50 stochastic passes)
**Uncertainty Status:** {flag}

---
### Class Probabilities
| Class | Probability |
|-------|------------|
| Glioma | {mc_mean[0]:.4f} ({mc_mean[0]*100:.1f}%) |
| Meningioma | {mc_mean[1]:.4f} ({mc_mean[1]*100:.1f}%) |
| No Tumor | {mc_mean[2]:.4f} ({mc_mean[2]*100:.1f}%) |
| Pituitary | {mc_mean[3]:.4f} ({mc_mean[3]*100:.1f}%) |

---
### Grad-CAM++ Explanation
The heatmap shows which regions influenced this prediction.
Red/yellow = high attention, blue = low attention.
"""
    return overlay, result_text, flag

# Quick test
print("\nTesting prediction function...")
test_img = np.random.randint(50, 200, (256,256,3), dtype=np.uint8)
overlay, result, flag = predict_mri(test_img)
print(f"Test passed — overlay shape: {overlay.shape}")
print(f"Flag: {flag}")
print("\nReady for Cell 3 — Gradio UI")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 765.1/765.1 kB 14.7 MB/s eta 0:00:00
Gradio version: 6.19.0
Last conv layer: top_conv
Uncertainty thresholds: green<0.05, red>0.15

Testing prediction function...
Test passed — overlay shape: (224, 224, 3)
Flag: 🟡 MODERATE UNCERTAINTY (std=0.0598) — Consider radiologist review

Ready for Cell 3 — Gradio UI


In [ ]:

# Full Gradio Interface
#  UI + uncertainty flag + clinical disclaimer
#

# Clinical disclaimer (Task 39)
DISCLAIMER = """
 **CLINICAL DISCLAIMER — READ BEFORE USE**

This system is a **research prototype only**, developed for academic demonstration purposes.
It is **NOT validated for clinical use** and must **NOT** be used to inform, replace, or
influence any medical diagnosis or treatment decision. All outputs must be interpreted
exclusively by qualified medical professionals with access to complete patient history,
imaging context, and clinical judgment. This tool does not meet regulatory requirements
for medical device classification in any jurisdiction.
"""

# CSS for colored uncertainty badge
custom_css = """
.uncertainty-high { background-color: #ff4444; color: white; padding: 8px; border-radius: 6px; }
.uncertainty-moderate { background-color: #ff9900; color: white; padding: 8px; border-radius: 6px; }
.uncertainty-low { background-color: #00aa44; color: white; padding: 8px; border-radius: 6px; }
"""

with gr.Blocks(title="Brain Tumor MRI Classifier", css=custom_css) as demo:

    # Header + disclaimer (Task 39)
    gr.Markdown("#  Brain Tumor MRI Classification")
    gr.Markdown("### EfficientNetB3 + Monte Carlo Dropout Uncertainty + Grad-CAM++ XAI")
    gr.Markdown(DISCLAIMER)

    gr.Markdown("---")

    # Model stats row
    gr.Markdown("""
    **Model Stats:** EfficientNetB3 | Test Accuracy: 91.44% | AUC: 0.9863 |
    MC Dropout: 50 passes | Trained on 5,600 CLAHE-preprocessed MRI images
    """)

    gr.Markdown("---")

    with gr.Row():
        # Left column: inputs
        with gr.Column(scale=1):
            gr.Markdown("### Upload MRI Image")
            image_input = gr.Image(
                type="numpy",
                label="MRI Scan (JPG/PNG)",
                height=300
            )
            predict_btn = gr.Button(
                " Analyze MRI",
                variant="primary",
                size="lg"
            )
            gr.Markdown("""
            **Supported classes:**
            - Glioma
            - Meningioma
            - No Tumor
            - Pituitary Adenoma
            """)

        # Right column: outputs
        with gr.Column(scale=2):
            gr.Markdown("### Results")

            with gr.Row():
                # Grad-CAM overlay
                gradcam_output = gr.Image(
                    label="Grad-CAM++ Explanation",
                    height=300
                )

            # Uncertainty flag (Task 38)
            gr.Markdown("### Uncertainty Status")
            uncertainty_flag = gr.Markdown(
                value="Upload an image and click Analyze to see results."
            )

            # Full results
            result_output = gr.Markdown(
                label="Prediction Details",
                value=""
            )

    gr.Markdown("---")

    # Example note
    gr.Markdown("""
    ### How to use
    1. Upload a brain MRI image (axial, sagittal, or coronal view)
    2. Click **Analyze MRI**
    3. Review the prediction, confidence score, and Grad-CAM++ heatmap
    4. Check the uncertainty flag — red means radiologist review is required
    5. Never use this output as a standalone diagnostic tool

    ### Uncertainty thresholds
    -  **Green** (std < 0.05): Model is confident means low uncertainty
    -  **Yellow** (0.05 ≤ std ≤ 0.15): Moderate uncertainty means consider expert review
    -  **Red** (std > 0.15): High uncertainty means radiologist review REQUIRED

    ### About the model
    This pipeline uses EfficientNetB3 fine-tuned on the Masoud Nickparvar Brain Tumor MRI
    Dataset (7,023 images, 4 classes). Uncertainty is estimated via Monte Carlo Dropout
    (50 stochastic forward passes). Explanations are generated via Grad-CAM++ on the
    final convolutional layer (`top_conv`). Trained and evaluated with SEED=42 for
    full reproducibility.

    [ GitHub Repository](https://github.com/AminahAsif/Mri-Brain-Tumor-uq-xai-fairness)
    """)


    gr.Markdown("*Developed by Amina Asif*")

    # Wire up button
    predict_btn.click(
        fn=predict_mri,
        inputs=[image_input],
        outputs=[gradcam_output, result_output, uncertainty_flag]
    )

print("Gradio interface built successfully.")
print("Launching demo...")

# Launch in Colab
demo.launch(
    share=True,         # generates public URL
    debug=False,
    quiet=True
)

Gradio interface built successfully.
Launching demo...
* Running on public URL: https://6c8ab366213b3299ad.gradio.live


In [ ]:

#  Save model + app files for HuggingFace
=
import os
import shutil

HF_DIR = '/content/hf_deployment'
os.makedirs(HF_DIR, exist_ok=True)

# ── 1. Save model in SavedModel format (smaller, HF compatible)
print("Saving model...")
effnet_model.save(f'{HF_DIR}/model.keras')
print(f"Model saved: {HF_DIR}/model.keras")

# ── 2. Write app.py (the HuggingFace Spaces entry point)
app_py = '''import gradio as gr
import numpy as np
import tensorflow as tf
import cv2
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

CLASS_NAMES = ["glioma", "meningioma", "notumor", "pituitary"]
IMG_SIZE    = (224, 224)
UNC_HIGH    = 0.15
UNC_LOW     = 0.05

# Load model
print("Loading model...")
effnet_model  = tf.keras.models.load_model("model.keras")
backbone      = effnet_model.layers[1]
last_conv     = [l.name for l in reversed(backbone.layers)
                 if isinstance(l, tf.keras.layers.Conv2D)][0]
grad_model    = tf.keras.Model(
    inputs=backbone.input,
    outputs=[backbone.get_layer(last_conv).output, backbone.output]
)
dropout_layer = effnet_model.layers[2]
head_layer    = effnet_model.layers[3]
print(f"Model loaded. Last conv: {last_conv}")

DISCLAIMER = """
 **CLINICAL DISCLAIMER — READ BEFORE USE**
This system is a **research prototype only**, developed for academic demonstration.
It is **NOT validated for clinical use** and must **NOT** be used to inform, replace,
or influence any medical diagnosis or treatment decision.
All outputs must be interpreted by qualified medical professionals.
"""

def preprocess_image(img):
    img_resized = cv2.resize(img, IMG_SIZE)
    img_pre = tf.keras.applications.efficientnet.preprocess_input(
                  img_resized.astype(np.float32))
    return img_resized, img_pre[np.newaxis]

def mc_dropout_predict(img_pre, n_passes=50):
    features = backbone(img_pre, training=False).numpy()
    feat_tf  = tf.constant(features)
    preds = []
    for _ in range(n_passes):
        drop = dropout_layer(feat_tf, training=True)
        pred = head_layer(drop, training=False).numpy()
        preds.append(pred[0])
    preds       = np.array(preds)
    mc_mean     = preds.mean(axis=0)
    mc_std      = preds.std(axis=0)
    uncertainty = float(mc_std.mean())
    confidence  = float(mc_mean.max())
    pred_class  = int(np.argmax(mc_mean))
    return mc_mean, uncertainty, confidence, pred_class

def make_gradcam(img_pre, pred_class, img_display):
    with tf.GradientTape() as t2:
        with tf.GradientTape() as t1:
            with tf.GradientTape() as t0:
                inp = tf.cast(img_pre, tf.float32)
                co, pred = grad_model(inp)
                t0.watch(co); t1.watch(co); t2.watch(co)
                loss = pred[:, pred_class]
            g1 = t0.gradient(loss, co)
        g2 = t1.gradient(g1, co)
    g3 = t2.gradient(g2, co)
    co=co[0]; g1=g1[0]; g2=g2[0]; g3=g3[0]
    s   = tf.reduce_sum(co, axis=(0,1))
    den = 2*g2 + s[None,None,:]*g3
    den = tf.where(den==0, tf.ones_like(den), den)
    w   = tf.reduce_sum((g2/den)*tf.nn.relu(g1), axis=(0,1))
    h   = tf.nn.relu(tf.reduce_sum(w*co, axis=-1)).numpy()
    h   = cv2.resize(h, IMG_SIZE)
    if h.max() > 0:
        h = (h-h.min())/(h.max()-h.min())
    colormap = matplotlib.colormaps["jet"]
    hc = (colormap(h)[:,:,:3]*255).astype(np.uint8)
    return cv2.addWeighted(img_display, 0.55, hc, 0.45, 0)

def predict_mri(image):
    if image is None:
        return None, "Please upload an MRI image.", ""
    img_display, img_pre = preprocess_image(image)
    mc_mean, unc, conf, pred_class = mc_dropout_predict(img_pre)
    overlay = make_gradcam(img_pre, pred_class, img_display)
    if unc > UNC_HIGH:
        flag = f" HIGH UNCERTAINTY (std={unc:.4f}) means Radiologist review REQUIRED"
    elif unc < UNC_LOW:
        flag = f" LOW UNCERTAINTY (std={unc:.4f}) means Model is confident"
    else:
        flag = f" MODERATE UNCERTAINTY (std={unc:.4f}) means Consider radiologist review"

    result = f"""## Prediction: {CLASS_NAMES[pred_class].upper()}
**Confidence:** {conf:.1%}
**MC Dropout Uncertainty:** {unc:.4f} (50 stochastic passes)
**Status:** {flag}

---
### Class Probabilities
| Class | Probability |
|-------|------------|
| Glioma | {mc_mean[0]:.4f} ({mc_mean[0]*100:.1f}%) |
| Meningioma | {mc_mean[1]:.4f} ({mc_mean[1]*100:.1f}%) |
| No Tumor | {mc_mean[2]:.4f} ({mc_mean[2]*100:.1f}%) |
| Pituitary | {mc_mean[3]:.4f} ({mc_mean[3]*100:.1f}%) |
"""
    return overlay, result, flag

with gr.Blocks(title="Brain Tumor MRI Classifier") as demo:
    gr.Markdown("#  Brain Tumor MRI Classification")
    gr.Markdown("### EfficientNetB3 + MC Dropout Uncertainty + Grad-CAM++ XAI")
    gr.Markdown(DISCLAIMER)
    gr.Markdown("---")
    gr.Markdown("**Model:** EfficientNetB3 | **Accuracy:** 91.44% | **AUC:** 0.9863 | **MC Dropout:** 50 passes")
    gr.Markdown("---")
    with gr.Row():
        with gr.Column(scale=1):
            image_input = gr.Image(type="numpy", label="Upload MRI Scan", height=300)
            predict_btn = gr.Button(" Analyze MRI", variant="primary", size="lg")
            gr.Markdown("**Classes:** Glioma | Meningioma | No Tumor | Pituitary")
        with gr.Column(scale=2):
            gradcam_output = gr.Image(label="Grad-CAM++ Explanation", height=300)
            uncertainty_flag = gr.Markdown("Upload an image and click Analyze.")
            result_output = gr.Markdown()
    gr.Markdown("---")
    gr.Markdown("""
### Uncertainty thresholds
-  **Green** (std < 0.05): Model is confident
-  **Yellow** (0.05 ≤ std ≤ 0.15): Moderate — consider expert review
-  **Red** (std > 0.15): High — radiologist review REQUIRED

###  This is NOT a clinical tool. For research demonstration only.
*Developed by Amina Asif *
""")
    predict_btn.click(
        fn=predict_mri,
        inputs=[image_input],
        outputs=[gradcam_output, result_output, uncertainty_flag]
    )

demo.launch()
'''

with open(f'{HF_DIR}/app.py', 'w') as f:
    f.write(app_py)

# ── 3. Write requirements.txt for HuggingFace
requirements = """gradio
tensorflow-cpu
opencv-python-headless
numpy
matplotlib
"""
with open(f'{HF_DIR}/requirements.txt', 'w') as f:
    f.write(requirements)

# ── 4. Write README for HuggingFace Space
hf_readme = """---
title: Brain Tumor MRI Classifier
emoji: 🧠
colorFrom: blue
colorTo: purple
sdk: gradio
sdk_version: "4.0"
app_file: app.py
pinned: false
---

# Brain Tumor MRI Classification
EfficientNetB3 + Monte Carlo Dropout Uncertainty + Grad-CAM++ XAI

**Test Accuracy:** 91.44% | **AUC:** 0.9863 | **MC Dropout:** 50 passes

 Research prototype only  NOT for clinical use.
"""
with open(f'{HF_DIR}/README.md', 'w') as f:
    f.write(hf_readme)

print("HuggingFace deployment files created:")
for f in os.listdir(HF_DIR):
    size = os.path.getsize(f'{HF_DIR}/{f}')
    print(f"  {f}: {size/1e6:.1f} MB" if size > 1e6 else f"  {f}: {size} bytes")

# Copy to Drive for permanent storage
hf_drive = f'{PROJECT_ROOT}/deployment'
os.makedirs(hf_drive, exist_ok=True)
for fname in ['app.py', 'requirements.txt', 'README.md']:
    shutil.copy(f'{HF_DIR}/{fname}', f'{hf_drive}/{fname}')

print(f"\nDeployment files saved to Drive: {hf_drive}")
print("\nNext step: Upload to HuggingFace Spaces (instructions in Cell 5)")

Saving model...
Model saved: /content/hf_deployment/model.keras
HuggingFace deployment files created:
  README.md: 370 bytes
  requirements.txt: 62 bytes
  app.py: 5580 bytes
  model.keras: 73.0 MB

Deployment files saved to Drive: /content/drive/MyDrive/MRI_Brain_Tumor_Project/deployment

Next step: Upload to HuggingFace Spaces (instructions in Cell 5)
